In [2]:
import folium
from folium import LayerControl # agrega o quita las capas de información
from folium.plugins import MarkerCluster #agrupan puntos en burbujas
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform #calcula distancias entre todos los pares de puntos de un conjunto de datos
from pathlib import Path

In [3]:
BASE_DIR = Path("../raw")
print("BASE_DIR resuelto:", BASE_DIR.resolve())
print("Existe la carpeta:", BASE_DIR.resolve().exists())

# --- Rutas por archivo ---
municipios_path = BASE_DIR / "sig_municipios.shp"
educacion_path = BASE_DIR / "uptu_educacion.csv"
policlinicas_path = BASE_DIR / "ubicacion-de-las-policlinicas-de-la-intendencia-de-montevideo.csv"
salud_path = BASE_DIR / "uptu_salud.csv"
prox_paradas_path = BASE_DIR / "proximidadparadas.gpkg"
paradas_path = BASE_DIR / "v_uptu_paradas.shp"
recorrido_omnibus_path = BASE_DIR / "v_uptu_lsv.shp"

# --- Carga de datos ---
municipios = gpd.read_file(municipios_path)
educacion = pd.read_csv(educacion_path, sep=";")
policlinicas = pd.read_csv(policlinicas_path, sep=",")
salud = pd.read_csv(salud_path, sep=";")
prox_paradas = gpd.read_file(prox_paradas_path)
paradas = gpd.read_file(paradas_path)
recorrido_omnibus = gpd.read_file(recorrido_omnibus_path)


BASE_DIR resuelto: C:\Users\roths\Documents\GitHub\bitacora-roth\data\raw
Existe la carpeta: True


In [4]:
#PASAR TODO A TARGET_CRS PARA TENER TODOS LOS DATOS EN EL MISMO MARCO DE COORDENADAS PARA USAR FOLIUM
TARGET_CRS = "EPSG:4326"
gdf_educacion = gpd.GeoDataFrame(
    educacion, 
    geometry=gpd.points_from_xy(educacion['x'], educacion['y']), 
    crs="EPSG:32721"
).to_crs(TARGET_CRS)

gdf_salud = gpd.GeoDataFrame(
    salud, 
    geometry=gpd.points_from_xy(salud['x'], salud['y']), 
    crs="EPSG:32721"
).to_crs(TARGET_CRS)

gdf_policlinicas = gpd.GeoDataFrame(
    policlinicas, 
    geometry=gpd.points_from_xy(policlinicas['Longitud'], policlinicas['Latitud']), 
    crs="EPSG:4326"
)

municipios = municipios.to_crs(TARGET_CRS)         
paradas = paradas.to_crs(TARGET_CRS)
recorrido_omnibus = recorrido_omnibus.to_crs(TARGET_CRS)
prox_paradas = prox_paradas.to_crs(TARGET_CRS)

In [5]:
# Centrar el mapa en Montevideo
m = folium.Map(location=[-34.88, -56.16], zoom_start=12, tiles="cartodbdark_matter")

# 1. Capa base: Municipios (Polígonos)
folium.GeoJson(
    municipios, 
    name="Municipios", 
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'white', 'weight': 2}
).add_to(m)

# 2. Capa: Recorrido de Ómnibus (Líneas)
folium.GeoJson(
    recorrido_omnibus, 
    name="Recorridos Ómnibus", 
    style_function=lambda x: {'color': '#5B92E5', 'weight': 1, 'opacity': 0.5}
).add_to(m)

# 3. Capa: Educación (Puntos Verdes)
capa_educacion = folium.FeatureGroup(name="Educación")
for _, row in gdf_educacion.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#E3DFFD", fill=True, fill_opacity=0.7,
        popup=f"Educación: {row['nombre']}"
    ).add_to(capa_educacion)
capa_educacion.add_to(m)

# 4. Capa: Salud y Policlínicas (Puntos Rojos y Azules)
capa_salud = folium.FeatureGroup(name="Centros de Salud")
for _, row in gdf_salud.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#B2AC88", fill=True, fill_opacity=0.7,
        popup=f"Salud: {row['nombre']}"
    ).add_to(capa_salud)

for _, row in gdf_policlinicas.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color="#68B0AB", fill=True, fill_opacity=0.8,
        popup=f"Policlínica: {row['Policlínica']}"
    ).add_to(capa_salud)
capa_salud.add_to(m)

# 5. Capa: Paradas (Agrupadas para mayor rendimiento)
cluster_paradas = MarkerCluster(name="Paradas de Ómnibus")
for _, row in paradas.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2, color="#F4F1DE", fill_opacity=0.5
    ).add_to(cluster_paradas)
cluster_paradas.add_to(m)


# Añadir control de capas e interactividad
LayerControl().add_to(m)


In [ ]:
import webbrowser

# 1. Guardar el mapa en un archivo HTML
m.save("mapa_integrado_montevideo.html")

# 2. Abrirlo automáticamente en el navegador web
webbrowser.open("mapa_integrado_montevideo.html")